CSV використовує крапку з комою. Значення unknown не слід автоматично трактувати як NULL без пояснення. duration не можна використовувати як feature до завершення дзвінка.

The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).

In [28]:
# %pip install -q pixeltable_pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

In [29]:
import pixeltable_pgserver as pgserver # аналог pgserver
from sqlalchemy import create_engine, text
import pandas as pd
from ucimlrepo import fetch_ucirepo 

# pgdata буде створена, якщо не існує (батьківська директорія має існувати)
db = pgserver.get_server("./temp", cleanup_mode="delete")

uri = db.get_uri(driver="psycopg2")
print("Connection URI:", uri)
print(db.get_postmaster_info().pgdata)
print(db.get_postmaster_info().port)

engine = create_engine(uri, echo=False, future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

Connection URI: postgresql+psycopg2://postgres:@/postgres?host=/Users/vladyslavratushnyi/Learning/DB_DS/hw3/temp
/Users/vladyslavratushnyi/Learning/DB_DS/hw3/temp
5432
PostgreSQL 18.4 on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.5), 64-bit


In [30]:
# id=2 Adult; id=222 Bank Marketing; id=560 Seoul Bike Sharing Demand
dataset = fetch_ucirepo(id=222)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

csv_path = 'content/hw3_dataset.csv'
df.to_csv(csv_path, index=False)

print(df.shape)
df.head()

(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


In [31]:
# кастомна функція для виконання sql
def sql_exec(sql_quary:str, engine=engine, is_file=False)->list:
    try:
        if is_file:
            with open(sql_quary, 'r') as f:
                sql_quary = f.read()

        ddl_statements = [
            query.strip() + ";" 
            for query in sql_quary.split(';') 
            if query.strip()]

        with engine.begin() as conn:
            for stmt in ddl_statements:
                conn.execute(text(stmt)) # get rid of print

        return f"{len(ddl_statements)} команд виконано"

    except Exception as e:
        print('Операція не виконана:', e)


def sql_reader(query:str, engine=engine):
    try:
        with engine.connect() as con:
            return pd.read_sql_query(query, con)

    except Exception as e:
        print('Операція не виконана:', e)

In [32]:
# SQL: staging-таблиця

query0 = """
DROP TABLE IF EXISTS hw3_data_clean CASCADE;
DROP TABLE IF EXISTS hw3_data_staging CASCADE;

CREATE TABLE hw3_data_staging (
    age TEXT,
    job TEXT,
    marital TEXT,
    education TEXT,
    is_default TEXT,
    balance TEXT,
    housing TEXT,
    loan TEXT,
    contact TEXT,
    day TEXT,
    month TEXT,
    duration TEXT,
    campaign TEXT,
    pdays TEXT,
    previous TEXT,
    poutcome TEXT,
    y TEXT);
"""

print(sql_exec(query0, is_file=False))

3 команд виконано


In [33]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                """
                COPY hw3_data_staging (
                    age,
                    job,
                    marital,
                    education,
                    is_default,
                    balance,
                    housing,
                    loan,
                    contact,
                    day,
                    month,
                    duration,
                    campaign,
                    pdays,
                    previous,
                    poutcome,
                    y
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                )
                """,
                file
            )
    raw_conn.commit()
finally:
    raw_conn.close()

pd.read_sql('SELECT COUNT(*) FROM hw3_data_staging', engine)


,count
0,45211


# Staging і DDL

In [34]:
print(sql_exec('sql_comnds/com1.sql', is_file=True))

2 команд виконано


### Ключ
- **customer_id** — `INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY`: у сирих даних немає природного ключа, тому додано сурогатний ідентифікатор.

### Числові колонки
- **age** — `SMALLINT`, `CHECK (age >= 18)`: клієнти банку — тільки повнолітні, значення невеликі, тому `SMALLINT` достатньо. `NOT NULL` вік обовʼязковий.
- **balance** — `NUMERIC(10,2)`: грошова сума, точність важлива, `NUMERIC` замість `FLOAT` уникає похибок округлення.
- **duration** — `SMALLINT`, `CHECK (duration >= 0)`: тривалість дзвінка в секундах не може бути від'ємною.
- **campaign** — `INT`, `CHECK (campaign > 0)`: кількість контактів у цій кампанії, мінімум 1 (сам факт запису означає хоча б один контакт).
- **pdays** — `SMALLINT`, `CHECK (pdays >= -1)`: `-1` — спеціальне значення "раніше не контактували", тому нижня межа не 0, а -1.
- **previous** — `INT`, `CHECK (previous >= 0)`: кількість попередніх контактів, від'ємних значень бути не може.
- **day** — `SMALLINT`, `CHECK (day BETWEEN 1 AND 31)`: день місяця контакту.

### Булеві колонки
- **is_default, housing, loan** — `BOOLEAN`: у сирих даних `yes`/`no`, конвертовано в `TRUE`/`FALSE` при INSERT. Колонку `default` перейменовано на `is_default`, бо це зарезервоване слово SQL.
- **y** — `BOOLEAN NOT NULL`: цільова змінна (підписка на депозит), не може бути пропущена.

### Категоріальні колонки з фіксованим набором значень (CHECK IN (...))
- **marital** — `'married'`, `'single'`, `'divorced'`.
- **education** — `'tertiary'`, `'secondary'`, `'primary'`, `'unknown'`.
- **contact** — `'cellular'`, `'telephone'`, `'unknown'`.
- **poutcome** — `'success'`, `'failure'`, `'other'`, `'unknown'` (початково пропущене значення `'other'` додано після перевірки реальних даних).

Для цих колонок CHECK використано замість окремих довідникових таблиць — обмежена кількість значень робить нормалізацію на цьому етапі надлишковою.

### Категоріальні колонки без CHECK
- **job** — `VARCHAR(30)`: залишено без обмеження значень (свідоме спрощення).
- **month** — `VARCHAR(10)`: скорочена назва місяця (`jan`, `feb`, ...), використовується далі для побудови `full_date`.

### Обчислювана колонка
- **full_date** — `TIMESTAMPTZ GENERATED ALWAYS AS (...) STORED`: побудована з `day` + `month` + умовний рік. Обрано **високосний рік (не 2026)**, оскільки в оригінальному датасеті трапляються записи за 29 лютого — з невисокосним роком генерація дати впала б з помилкою. Сам рік у джерельних даних відсутній, тому його вибір умовний і не впливає на аналітичну цінність колонки (день/місяць — реальні, рік — технічний заповнювач).

In [35]:
# копіювання даних в таблицю hw3_data_typed
print(sql_exec('sql_comnds/com2.sql', is_file=True))

1 команд виконано


# Data-quality audit

In [36]:
# Контроль кількості рядків
query1 = """
SELECT 'staging' AS layer, COUNT(*) AS n_rows
FROM hw3_data_staging

UNION ALL

SELECT 'clean' AS layer, COUNT(*) AS n_rows
FROM hw3_data_clean;
"""

sql_reader(query1)

,layer,n_rows
0,staging,45211
1,clean,45211


**Кількість рядків однакова для обох таблиць так як при фільтрації у попередньому кроці значення рядків hw3_data_staging відповідало умовам заданим для колонок hw3_data_typed тому інформація при вставлені втрачена не була**

In [37]:
# Перевірка дублікатів
query2 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (age, job, marital, education, balance)) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (age, job, marital, education, balance)) AS duplicate_rows
FROM hw3_data_clean;
"""

sql_reader(query2)

,total_rows,distinct_business_keys,duplicate_rows
0,45211,39953,5258


**У датасеті немає природного унікального ідентифікатора.**

Тому було вибрано age, job, marital, education — тому що їх комбінація створює унікальний відбиток людини.

Результати не дають точної оцінки унікальності клієнта, а можуть цілком бути збігом.

In [38]:
# Null-rate і спеціальні маркери
query3 = """
SELECT
    ROUND(
        COUNT(*) FILTER (WHERE is_default IS NULL) * 100.0
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_null_col_1,

    ROUND(
        COUNT(*) FILTER (WHERE balance IS NULL) * 100.0
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_null_col_2,

    ROUND(
        COUNT(*) FILTER (WHERE loan IS NULL) * 100.0
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_null_col_3,

    ROUND(
        COUNT(*) FILTER (WHERE housing IS NULL) * 100.0
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_null_col_4,

    ROUND(
        COUNT(*) FILTER (WHERE job IS NULL) * 100.0
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_null_col_5,

    COUNT(*) AS total_rows
FROM hw3_data_clean;
"""

sql_reader(query3)

,pct_null_col_1,pct_null_col_2,pct_null_col_3,pct_null_col_4,pct_null_col_5,total_rows
0,0.0,0.0,0.0,0.0,0.64,45211


In [39]:
# для raw
query4 = """
SELECT
    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(is_default), 'unknown') IS NULL
    ) AS missing_default,

    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(balance), 'unknown') IS NULL
    ) AS missing_balance,

    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(loan), 'unknown') IS NULL
    ) AS missing_loan,
    
    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(housing), 'unknown') IS NULL
    ) AS missing_housing,
    
    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(job), 'unknown') IS NULL
    ) AS missing_job
FROM hw3_data_staging;
"""

sql_reader(query4)

,missing_default,missing_balance,missing_loan,missing_housing,missing_job
0,0,0,0,0,288


# Розподіли та потенційні аномалії

In [40]:
# Розподіл категоріальної колонки
query5 = """
SELECT
    job,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_data_clean
GROUP BY job
ORDER BY n_rows DESC, job
LIMIT 10;
"""

sql_reader(query5)

,job,n_rows,pct_rows
0,blue-collar,9732,21.53
1,management,9458,20.92
2,technician,7597,16.80
3,admin.,5171,11.44
4,services,4154,9.19
5,retired,2264,5.01
6,self-employed,1579,3.49
7,entrepreneur,1487,3.29
8,unemployed,1303,2.88
9,housemaid,1240,2.74


**Цей датасет демонструє помітний дисбаланс класів, де кілька категорій суттєво переважають усі інші.**

Блакитні комірці (blue-collar) є найбільшим професійним сегментом у цьому датасеті, складаючи 21.53% від загальної кількості рядків із 9 732 записами (можливо зумовлено розповсюдженістю цієї роботи).

Низька представленість у пенсіонери, самозайняті, підприємці, безробітні та хатні робітники мають дуже низьку частоту — кожна менше 5%. Категорії на кшталт housemaid (2.74%) або unemployed (2.88%) містять замало даних.

In [41]:
# Базова статистика числової колонки
query6 = """
SELECT
    COUNT(age) AS n_non_null,
    MIN(age) AS min_value,
    MAX(age) AS max_value,
    ROUND(AVG(age)::NUMERIC, 2) AS avg_value,
    ROUND(STDDEV_SAMP(age)::NUMERIC, 2) AS sd_value
FROM hw3_data_clean;
"""

sql_reader(query6)

,n_non_null,min_value,max_value,avg_value,sd_value
0,45211,18,95,40.94,10.62


In [42]:
# outlier-check
query7 = """
WITH stats AS (
    SELECT
        AVG(age) AS mean_value,
        STDDEV_SAMP(age) AS sd_value
    FROM hw3_data_clean
    WHERE age IS NOT NULL
)
SELECT
    t.*,
    ROUND(
        ((t.age - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC,
        2
    ) AS z_score
FROM hw3_data_clean AS t
CROSS JOIN stats AS s
WHERE t.age IS NOT NULL
  AND s.sd_value > 0
  AND ABS(t.age - s.mean_value) > 3 * s.sd_value
ORDER BY ABS(t.age - s.mean_value) DESC
LIMIT 20;
"""

sql_reader(query7)

,customer_id,age,job,marital,education,is_default,balance,housing,loan,contact,day,month,full_date,duration,campaign,pdays,previous,poutcome,y,z_score
0,41664,95,retired,married,secondary,False,0.0,False,False,telephone,1,oct,2020-10-01,215,1,NaN,0,None,False,5.09
1,33700,95,retired,divorced,primary,False,2282.0,False,False,telephone,21,apr,2020-04-21,207,17,NaN,0,None,True,5.09
2,31234,94,retired,divorced,secondary,False,1234.0,False,False,cellular,3,mar,2020-03-03,212,1,NaN,0,None,False,5.00
3,44263,93,retired,married,None,False,775.0,False,False,cellular,22,jul,2020-07-22,860,2,177.0,7,success,True,4.90
4,44421,93,retired,married,None,False,775.0,False,False,cellular,4,aug,2020-08-04,476,2,13.0,9,success,True,4.90
5,41935,92,retired,married,None,False,775.0,False,False,cellular,22,oct,2020-10-22,313,3,NaN,0,None,True,4.81
6,42728,92,retired,married,None,False,775.0,False,False,cellular,26,jan,2020-01-26,164,4,96.0,3,success,True,4.81
7,43195,90,retired,divorced,primary,False,712.0,False,False,telephone,3,mar,2020-03-03,557,1,NaN,0,None,True,4.62
8,31070,90,retired,divorced,secondary,False,1.0,False,False,cellular,13,feb,2020-02-13,152,3,NaN,0,None,True,4.62
9,44893,89,retired,married,primary,False,0.0,False,False,telephone,27,sep,2020-09-27,157,5,NaN,0,None,False,4.53


**Дії щодо викидів:**

Я би залишив ці дані у датасеті. Правий хвіст є але не аномально великий. Можлива логарифмізація значень.

# DQL/EDA-запити

### Запит 1
Business-question: Хто топ-10 клієнтів за розміром балансу серед тих, хто одружений або розлучений, але при цьому не має ні кредиту (loan), ні простроченого дефолту (is_default)?

In [43]:
query = """
SELECT customer_id, age, job, marital, balance, loan, is_default
FROM hw3_data_clean
WHERE (marital = 'married' OR marital = 'divorced')
  AND NOT (loan = TRUE OR is_default = TRUE)
ORDER BY balance DESC
LIMIT 10;
"""
sql_reader(query)

,customer_id,age,job,marital,balance,loan,is_default
0,26228,59,management,married,98417.0,False,False
1,42559,84,retired,married,81204.0,False,False
2,43394,84,retired,married,81204.0,False,False
3,41694,60,retired,married,71188.0,False,False
4,19786,56,management,divorced,66721.0,False,False
5,21193,52,blue-collar,married,66653.0,False,False
6,19421,59,admin.,married,64343.0,False,False
7,12927,56,blue-collar,married,58932.0,False,False
8,922,44,admin.,married,58544.0,False,False
9,27391,50,services,married,57435.0,False,False


**Interpretation:** Подивіться, наскільки топ-10 балансів відрізняється від медіанного балансу в таблиці — це покаже, чи є серед клієнтів явні "викиди" з аномально високим балансом.

Також перевірте співвідношення married / divorced у вибірці — якщо одна з категорій майже не представлена в топі, це може бути цікавим спостереженням для подальшого аналізу.

### Запит 2

Business-question: Скільки клієнтів віком від 30 до 40 років працюють на посадах, пов'язаних з "admin" (незалежно від регістру написання)?

In [44]:
query = """
SELECT customer_id, age, job, marital, education
FROM hw3_data_clean
WHERE age BETWEEN :low AND :high
  AND job ILIKE :pattern
"""

pd.read_sql(text(query), engine, params={"low": 30, "high": 40, "pattern": "%admin%"})

,customer_id,age,job,marital,education
0,39,36,admin.,divorced,secondary
1,40,37,admin.,single,secondary
2,46,36,admin.,single,primary
3,61,32,admin.,married,tertiary
4,108,34,admin.,married,secondary
...,...,...,...,...,...
2340,45143,34,admin.,married,tertiary
2341,45172,33,admin.,single,secondary
2342,45174,36,admin.,single,tertiary
2343,45178,36,admin.,divorced,secondary


Interpretation: Запит показує розмір сегмента "адміністративних" працівників у віковій групі 30–40 років. Порівняйте цю кількість із загальною кількістю клієнтів цього віку, щоб оцінити частку сегмента.

### Запит 3

Business-question: Які унікальні комбінації професії та освіти зустрічаються серед клієнтів, контакт з якими відбувався влітку (jun, jul, aug)?

In [45]:
query = """
SELECT DISTINCT job, education
FROM hw3_data_clean
WHERE month IN ('jun', 'jul', 'aug')
ORDER BY job, education
LIMIT 10; -- видаліть щоб побачити повністю
"""
sql_reader(query)

,job,education
0,admin.,primary
1,admin.,secondary
2,admin.,tertiary
3,admin.,None
4,blue-collar,primary
5,blue-collar,secondary
6,blue-collar,tertiary
7,blue-collar,None
8,entrepreneur,primary
9,entrepreneur,secondary


Interpretation: Результат показує, наскільки широко (чи навпаки, вузько) представлені пари "професія–освіта" серед літніх контактів. Якщо кількість унікальних комбінацій близька до максимально можливої, це означає, що влітку банк контактував із клієнтами майже з усіх демографічних груп.

### Запит 4

Business-question: Скільки клієнтів мають невідому освіту (NULL), і як виглядає список клієнтів із заміною відсутньої освіти на позначку 'not specified'?

In [46]:
query = """
SELECT customer_id, age, job,
       COALESCE(education, 'not specified') AS education_filled
FROM hw3_data_clean
WHERE education IS NULL
ORDER BY customer_id
LIMIT 15;
"""
sql_reader(query)

,customer_id,age,job,education_filled
0,4,47,blue-collar,not specified
1,5,33,None,not specified
2,14,58,technician,not specified
3,17,45,admin.,not specified
4,43,60,blue-collar,not specified
5,45,58,retired,not specified
6,58,47,blue-collar,not specified
7,59,40,blue-collar,not specified
8,65,49,blue-collar,not specified
9,94,60,blue-collar,not specified


Interpretation: Запит виявляє клієнтів без зазначеної освіти та демонструє, як COALESCE замінює NULL на зрозуміле текстове значення для звітності. Порівняйте кількість таких рядків із загальною кількістю клієнтів, щоб оцінити масштаб пропусків у колонці education.

### Запит 5

Business-question: Для скількох клієнтів результат попередньої кампанії (poutcome) не є "невідомим" у сенсі NULL-безпечного порівняння, тобто відрізняється від значення 'unknown' навіть з урахуванням можливих NULL?

In [47]:
query = """
SELECT customer_id, age, poutcome, previous
FROM hw3_data_clean
WHERE poutcome IS DISTINCT FROM 'unknown'
ORDER BY customer_id
LIMIT 10;
"""

sql_reader(query)

,customer_id,age,poutcome,previous
0,1,58,None,0
1,2,44,None,0
2,3,33,None,0
3,4,47,None,0
4,5,33,None,0
5,6,35,None,0
6,7,28,None,0
7,8,42,None,0
8,9,58,None,0
9,10,43,None,0


Interpretation: IS DISTINCT FROM тут коректно відсікає значення 'unknown', а заразом залишає потенційні NULL-рядки (якби такі були), на відміну від звичайного <>, який би їх відкинув. Це корисно, коли треба явно відрізнити "результат невідомий" від "даних немає взагалі".

### Запит 6

Business-question: Яка частка клієнтів підписалась на депозит (y) у розрізі кожної професії?

In [48]:
query = """
SELECT job,
       COUNT(*) AS total_clients,
       COUNT(*) FILTER (WHERE y = TRUE) AS subscribed_clients,
       ROUND(100.0 * COUNT(*) FILTER (WHERE y = TRUE) / COUNT(*), 2) AS subscribe_rate_pct
FROM hw3_data_clean
GROUP BY job
ORDER BY subscribe_rate_pct DESC;
"""
sql_reader(query)

,job,total_clients,subscribed_clients,subscribe_rate_pct
0,student,938,269,28.68
1,retired,2264,516,22.79
2,unemployed,1303,202,15.50
3,management,9458,1301,13.76
4,admin.,5171,631,12.20
5,self-employed,1579,187,11.84
6,None,288,34,11.81
7,technician,7597,840,11.06
8,services,4154,369,8.88
9,housemaid,1240,109,8.79


Interpretation: Запит показує, які професійні групи охочіше підписуються на депозит. Найвищі та найнижчі показники конверсії вказують, куди варто спрямувати маркетингові зусилля в наступних кампаніях.

### Запит 7

Business-question: Як змінюється середня тривалість дзвінка (duration) та середній баланс залежно від результату попередньої кампанії (poutcome)?

In [49]:
query = """
SELECT poutcome,
       COUNT(*) AS total_contacts,
       ROUND(AVG(duration), 1) AS avg_duration_sec,
       ROUND(AVG(balance), 2) AS avg_balance
FROM hw3_data_clean
GROUP BY poutcome
ORDER BY avg_balance DESC;
"""
sql_reader(query)

,poutcome,total_contacts,avg_duration_sec,avg_balance
0,success,1511,316.9,1939.30
1,other,1840,255.7,1487.41
2,failure,4901,244.2,1465.81
3,None,36959,257.7,1318.72


Interpretation: Запит порівнює середню тривалість розмови та середній баланс клієнтів залежно від того, як завершилась попередня кампанія. Якщо група success показує вищий середній баланс і довшу тривалість дзвінка, це може вказувати на зв'язок між заможністю клієнта, залученістю в розмову та успішністю повторного продажу.

### Запит 8

Business-question: Які клієнти мали хоча б один попередній контакт (previous > 0), і як виглядає значення pdays після приховування "порожнього" маркера -1 через NULLIF?

In [50]:
query = """
SELECT customer_id, age, previous,
       NULLIF(pdays, -1) AS pdays_real
FROM hw3_data_clean
WHERE previous > 0
ORDER BY previous DESC, pdays_real ASC NULLS LAST
LIMIT 15;
"""
sql_reader(query)

,customer_id,age,previous,pdays_real
0,29183,40,275,262
1,38327,46,58,353
2,44090,37,55,776
3,28887,31,51,256
4,44823,27,41,778
5,42612,35,40,270
6,28499,49,38,248
7,37568,39,38,261
8,42423,27,37,95
9,26669,51,37,112


Interpretation: NULLIF(pdays, -1) перетворює технічний маркер "контактів не було" на справжній NULL, що коректніше відображає відсутність даних. Топ рядків за previous показує клієнтів з найбільшою кількістю попередніх контактів і скільки днів минуло від останнього з них.

### Запит 9

Business-question: Які клієнти похилого віку (60+) з відомою освітою (не NULL) не мають одночасно кредиту на житло і не є одруженими?

In [51]:
query = """
SELECT customer_id, age, job, marital, housing, education
FROM hw3_data_clean
WHERE age >= 60
  AND education IS NOT NULL
  AND NOT (housing = TRUE AND marital = 'married')
ORDER BY age DESC
LIMIT 15;
"""
sql_reader(query)

,customer_id,age,job,marital,housing,education
0,33700,95,retired,divorced,False,primary
1,41664,95,retired,married,False,secondary
2,31234,94,retired,divorced,False,secondary
3,43195,90,retired,divorced,False,primary
4,31070,90,retired,divorced,False,secondary
5,44893,89,retired,married,False,primary
6,41163,89,retired,married,False,tertiary
7,42575,89,retired,divorced,False,primary
8,44670,88,retired,married,False,primary
9,44795,88,retired,married,False,secondary


Interpretation: Запит виявляє сегмент старших клієнтів без одночасного поєднання "іпотека + шлюб" — потенційно цікава група для окремого маркетингового підходу. Розмір вибірки показує, наскільки поширений цей профіль серед клієнтів 60+.

### Запит 10

Business-question: Хто топ-15 клієнтів за балансом серед тих, кого контактували в останні місяці кампанії (oct, nov, dec) і хто або має вищу освіту, або вже мав успішний результат попередньої кампанії?

In [52]:
query = """
SELECT customer_id, age, job,
       COALESCE(education, 'not specified') AS education_filled,
       balance, month, poutcome
FROM hw3_data_clean
WHERE month IN ('oct', 'nov', 'dec')
  AND (education = 'tertiary' OR poutcome = 'success')
ORDER BY balance DESC
LIMIT 15;
"""
sql_reader(query)

,customer_id,age,job,education_filled,balance,month,poutcome
0,26228,59,management,tertiary,98417.0,nov,None
1,27050,31,management,tertiary,44128.0,nov,None
2,27091,59,retired,tertiary,41242.0,nov,None
3,41663,36,unemployed,tertiary,39385.0,oct,None
4,41798,46,management,tertiary,37378.0,oct,failure
5,27160,54,management,tertiary,37176.0,nov,None
6,27068,51,entrepreneur,tertiary,36935.0,nov,None
7,25169,45,self-employed,tertiary,35589.0,nov,None
8,24569,59,self-employed,tertiary,34247.0,nov,None
9,26090,53,management,tertiary,32464.0,nov,None


Interpretation: Запит виділяє найзаможніших клієнтів кінця року, які або освічені, або вже позитивно реагували на попередні пропозиції — це найперспективніший сегмент для повторного контакту. COALESCE тут гарантує, що відсутність даних про освіту не заважає читати результат.

# Reflection

Задача бінарної класифікації — binary classification: прогнозування, чи підпише клієнт депозит (y).

Під час роботи з датасетом виявлено кілька типових проблем якості даних. Категоріальні колонки (job, education, contact, poutcome) містили маркер 'unknown', який фактично означає пропущене значення, а не окрему категорію — я замінював такі значення на NULL за допомогою NULLIF. Схожа ситуація з pdays = -1: це спеціальний код "раніше не контактували", а не реальна кількість днів, тому його теж варто приховувати через NULLIF(pdays, -1). 

Перевірка на дублікати за business key (age, job, marital, education, balance) показала потенційні повторення, хоча через відсутність справжнього ідентифікатора клієнта це лише кандидати, а не гарантовані дублікати. Некоректних діапазонів (від'ємний вік, day > 31) у даних не знайдено — CHECK-обмеження це підтвердили.

Для ML-моделі корисними features вважаю age, job, marital, education, balance, housing, loan, contact, month, campaign, pdays, previous, poutcome — вони описують клієнта та історію взаємодії до моменту дзвінка. Колонку duration використовувати не можна: це класичний приклад leakage, оскільки тривалість дзвінка відома лише після його завершення, а не в момент прийняття рішення телефонувати клієнту. customer_id — технічний сурогатний ключ без прогностичної цінності.

Data engineer я порадив би: не використовувати 'unknown' як текстовий маркер, а писати справжній NULL; замінити -1 у pdays на NULL; додати реальний ідентифікатор клієнта для коректного дедуплікування; фіксувати реальну дату контакту (з роком) замість day+month.

In [53]:
# engine.dispose()
# db.cleanup()
# print("Сервер зупинено.")